# Analytical Window Functions: Rankings, Lead/Lag, and Moving Averages

**Module**: 03_dataframe_and_spark_sql  
**Notebook**: 06_window_functions.ipynb  
**Environment**: Apache Spark 4.2.1 · Standalone Cluster · PySpark · Python 3.13 · Java 21  

---

## Learning Objectives

- **What will I learn?**: How to construct Window specifications, compute ranking functions (row_number, rank, dense_rank), analytical offsets (lead, lag), and cumulative running totals.
- **Why does this topic exist?**: Window functions solve advanced analytical questions (e.g. top N items per group, year-over-year changes, 7-day rolling averages) without self-joins.
- **What problem does it solve?**: Performing group-based ranking with self-joins causes severe performance degradation.
- **Where is it used in real-world Data Engineering?**: Financial reporting, fraud detection (time between transactions), and sales leaderboards.

---

<!-- curriculum-enrichment:v1 -->

## Learning Contract

| Attribute | Value |
|---|---|
| **Priority** | Essential |
| **Estimated effort** | 3–4 h |
| **Prerequisites** | Joins, aggregations, Spark SQL |
| **Required depth** | Deep working knowledge: can write ranking, running, and time-based windows |
| **Data Engineer** | Use `partitionBy`/`orderBy`/frames correctly for ranking, running totals, and lag/lead. |
| **Senior Data Engineer** | Tune window partitions, frames, and memory for large analytical queries. |
| **Spark specialist** | Explain the window operator, sort, and frame evaluation. |
| **Next topic** | `03_dataframe_and_spark_sql/07_nulls_and_data_quality.ipynb` |

> **How to use this notebook**: read the problem first, then run the code, then connect the
> mechanism to the real-world cases and production notes. The goal is not to memorize API
> syntax; it is to be able to explain and defend a design decision.

---

## Problem and Motivation

### Why this concept exists

A window function computes a value for each row from a related set of rows without collapsing them. That
makes it ideal for rankings, running totals, and previous/next comparisons. The cost is a shuffle by the window
partition and a sort by the ordering key; an unbounded frame can also keep a large amount of state in memory.

### What happens if you ignore it

A team can still write code that runs on small data. The failure appears later: a job that
looks correct becomes slow, unstable, or expensive at scale, and the team lacks the vocabulary
to diagnose which Spark mechanism is responsible. This notebook builds that vocabulary and
ties it to observable behavior in the lab.

### A small mental model

Think of the topic as a contract between the logical work you describe and the physical work
Spark performs. The contract has a cost, a failure mode, and a set of trade-offs. The sections
below make each of those explicit.

---

## Core Concepts

### Definitions

| Term | Plain-language meaning | Technical meaning |
|---|---|---|
| **Window spec** | Defines the related rows | `partitionBy`, `orderBy`, frame |
| **Partition** | Groups rows for the window | Co-located on one task |
| **Frame** | Rows considered per row | Rows/range between boundaries |
| **`row_number`** | Sequential number | No ties; deterministic with a full ordering |
| **`rank`/`dense_rank`** | Ranking with ties | Gaps vs no gaps |
| **`lag`/`lead`** | Previous/next row | Useful for deltas and sessionization |

### How it works, step by step

1. The window operator shuffles data by the partition keys.
2. Each partition is sorted by the ordering keys.
3. The frame is evaluated for each row.
4. Unbounded frames may buffer many rows.
5. The result is joined back to the original row position.
6. Skew in partition keys creates stragglers.

### What to keep in mind

- The mechanism is not free; it consumes CPU, memory, network, or disk.
- The right choice depends on data size, distribution, latency target, and failure tolerance.
- A plan or UI observation is stronger evidence than an intuition about what "should" happen.

## 2. Window Function Architecture

```
Window Specification:
Window.partitionBy("category").orderBy(desc("revenue")).rowsBetween(start, end)
  |                                 |                       |
  v Segregates data                 v Sorts inside frame    v Defines frame slice
[Electronics]                   [Highest -> Lowest]   [e.g. -3 to Current Row]
  |-- Row 1: Laptop ($1500)       -> row_number=1, rank=1, dense_rank=1
  |-- Row 2: Phone ($800)         -> row_number=2, rank=2, dense_rank=2
[Clothing]
  |-- Row 1: Jacket ($200)        -> row_number=1, rank=1, dense_rank=1
```

### 1. Initialize SparkSession on Standalone Cluster

We connect to `spark://spark-master:7077` with the driver configured to bind to container hostname `jupyter`.

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *
import time

spark = (
    SparkSession.builder
    .appName("Window_Functions_Lab")
    .master("spark://spark-master:7077")
    .config("spark.driver.host", "jupyter")
    .config("spark.driver.bindAddress", "0.0.0.0")
    .config("spark.eventLog.enabled", "true")
    .config("spark.eventLog.dir", "file:///opt/spark-events")
    .config("spark.sql.warehouse.dir", "file:///opt/spark-data/warehouse")
    .config("spark.hadoop.fs.permissions.umask-mode", "000")
    .config("spark.sql.shuffle.partitions", "8")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")

print(f" SparkSession initialized successfully!")
print(f"Spark Version:      {spark.version}")
print(f"Master URL:         {spark.sparkContext.master}")
print(f"App Name:           {spark.sparkContext.appName}")
print(f"Default Parallelism: {spark.sparkContext.defaultParallelism}")
print(f"Spark UI available at: http://localhost:4040")

## 3. Practical: Top 2 Highest-Grossing Orders per Channel

In [ ]:
from pyspark.sql.window import Window

orders = spark.read.option("header", True).option("inferSchema", True).csv("/opt/spark-data/retail/orders.csv")

# Define window specification
channel_window = Window.partitionBy("channel").orderBy(F.desc("quantity"))

ranked_orders = (
    orders
    .withColumn("row_num", F.row_number().over(channel_window))
    .withColumn("rank_num", F.rank().over(channel_window))
    .withColumn("dense_rank_num", F.dense_rank().over(channel_window))
    .filter(F.col("row_num") <= 2)
    .select("channel", "order_id", "customer_id", "quantity", "row_num", "rank_num", "dense_rank_num")
)

ranked_orders.show()

## 4. Analytical Offsets: `lag()` to Calculate Days Between Customer Orders

In [ ]:
customer_window = Window.partitionBy("customer_id").orderBy("order_ts")

order_intervals = (
    orders
    .withColumn("prev_order_ts", F.lag("order_ts", 1).over(customer_window))
    .withColumn("days_since_prev", F.datediff(F.col("order_ts"), F.col("prev_order_ts")))
    .filter(F.col("prev_order_ts").isNotNull())
    .select("customer_id", "order_id", "order_ts", "prev_order_ts", "days_since_prev")
)

order_intervals.show(5)

## 5. Running Totals and Moving Averages with `rowsBetween`

In [ ]:
# Cumulative revenue running sum per customer
running_window = Window.partitionBy("customer_id").orderBy("order_ts").rowsBetween(Window.unboundedPreceding, Window.currentRow)

running_totals = (
    orders
    .withColumn("order_value", F.round(F.col("quantity") * F.col("unit_price"), 2))
    .withColumn("cumulative_spend", F.round(F.sum("order_value").over(running_window), 2))
    .select("customer_id", "order_ts", "order_value", "cumulative_spend")
)

running_totals.show(6)

---

## Experiment: rowsBetween vs rangeBetween Semantics

- **1. What are we testing?**: Demonstrate the difference between physical row offset (rowsBetween) and logical value range (rangeBetween).
- **2. What do we expect?**: rowsBetween counts physical row positions; rangeBetween groups tied order values.

In [ ]:
sample_data = spark.createDataFrame([
    ("A", 1, 100),
    ("A", 2, 100),
    ("A", 3, 200)
], ["user", "time_step", "value"])

w_rows = Window.partitionBy("user").orderBy("value").rowsBetween(Window.unboundedPreceding, Window.currentRow)
w_range = Window.partitionBy("user").orderBy("value").rangeBetween(Window.unboundedPreceding, Window.currentRow)

sample_data.withColumn("sum_rows", F.sum("value").over(w_rows)) \
           .withColumn("sum_range", F.sum("value").over(w_range)) \
           .show()

**3. What happened?**: For row 1 and 2 with tied value 100, sum_range immediately jumped to 200 for both rows, while sum_rows accumulated row by row.  
**4. Why did it happen?**: rangeBetween treats duplicate values in the orderBy column as peers within the same frame slice.  
**5. Practical Engineering Lesson**: Use rowsBetween when you need strict row-based moving averages (e.g. 3-row moving average).

---

## Real-World Use Cases

> The scenarios below are illustrative. Company names are omitted, and any numbers are hypothetical unless they are measured in this lab.

### Use Case 1: Top products per category

- **Industry**: Retail
- **Scenario**: A retailer ranks products by revenue within each category every week.
- **Problem**: A global ranking is wrong; the business needs a per-category ranking with ties handled correctly.
- **Why this concept is relevant**: `row_number`/`rank` over a category window computes the ranking without collapsing rows.
- **Implementation idea**: Aggregate revenue by product/category, apply `row_number` over `partitionBy(category).orderBy(revenue desc)`, and filter rank ≤ 10.
- **Expected result**: A top-10 list per category with deterministic tie-breaking.
- **Alternative**: A warehouse `QUALIFY`/window query.
- **Trade-offs**: Spark scales the window over lake data; a warehouse is simpler for curated marts.

### Use Case 2: Transaction velocity for fraud review

- **Industry**: Banking
- **Scenario**: A risk team flags accounts with many transactions in a short window.
- **Problem**: A simple daily count misses bursts that happen within minutes.
- **Why this concept is relevant**: A window over account ordered by time with a range frame computes rolling counts and amounts.
- **Implementation idea**: Use `rangeBetween(-3600, 0)` on epoch seconds, compute count/sum, and flag thresholds for analyst review.
- **Expected result**: A ranked list of suspicious bursts for investigation (not an automatic fraud decision).
- **Alternative**: A streaming rules engine or feature store.
- **Trade-offs**: Batch windows are good for historical analysis; streaming is needed for real-time blocking.

---

## Business Value

- Enables ranking and time-based analysis without losing row detail.
- Supports customer behavior and anomaly detection.
- Can replace complex self-joins with a single window operation.
- Requires careful frame and partition design to control memory.

> **Note on numbers**: this repository does not claim production benchmarks. If you want to
> quantify a business case, measure the job in your own environment (runtime, shuffle bytes,
> spill, infrastructure cost) before and after the change.

---

## Production Engineering

### What changes when this runs in production

- Always define the window partition and ordering explicitly.
- Use `row_number` with a deterministic tie-breaker for deduplication.
- Bound frames where possible; unbounded frames can grow state.
- Watch for skew in window partition keys.
- Validate results on a small sample before running at scale.

### Investigation checklist

- [ ] Open the Spark UI and identify the relevant tab for this topic.
- [ ] Compare the metric before and after a configuration or code change.
- [ ] Check driver and executor logs for warnings or retries.
- [ ] Confirm that the fix is measured, not assumed.

### Common production mistakes

1. Copying a local-lab setting into a production cluster without sizing it.
2. Optimizing a single job without checking the downstream SLA and data-quality impact.
3. Treating a successful run as proof of correctness; verify row counts, keys, and invariants.
4. Forgetting that retries and backfills must be idempotent.

### Experiment: row_number vs rank vs dense_rank

**Steps**

1. Create a small dataset with tied values.
2. Apply all three functions over the same window.
3. Compare the output rows.

**What to observe**: `row_number` breaks ties arbitrarily unless the ordering is complete; `rank` leaves gaps; `dense_rank` does not.

**Healthy result**: You can choose the right function for a business rule.

**Problematic result**: Using `row_number` without a tie-breaker produces non-deterministic results.

**Engineering takeaway**: Define the ordering completely.

---

## Trade-offs and Alternatives

### Advantages

- Expressive analytics
- Avoids self-joins
- Works with SQL and DataFrames

### Limitations

- Shuffle + sort cost
- Unbounded frames can use large memory
- Tie semantics can surprise

### When to use it

- Rankings
- Running totals
- Lag/lead
- Deduplication
- Fraud velocity

### When not to use it

- Huge unpartitioned windows
- Unbounded frames on high-cardinality keys
- Assuming stable order without a tie-breaker

### Alternatives inside Spark

- `groupBy` + join
- SQL `QUALIFY` patterns

### Alternatives outside Spark

- Warehouse window functions
- Streaming window operators

### Performance implications

Every choice has a cost. Measure runtime, shuffle read/write, spill, GC time, and task
duration percentiles before and after the change. A local four-core cluster is useful for
observing mechanics, but it is not a production benchmark.

---

## Practical Exercises

### Exercise 1 (Beginner): Top 1 Spending Customer per Region
**Problem Statement**: Join customers and orders, and find the single highest-spending customer in each region using row_number().  
**Input / Context**: customers and orders

In [ ]:
# Exercise 1: Top customer per region

### Exercise 2 (Intermediate): Percentage of Total Spend per Customer
**Problem Statement**: Calculate each individual order's percentage contribution to that customer's lifetime total spend.  
**Input / Context**: orders DataFrame

In [ ]:
# Exercise 2: Spend contribution

### Exercise 3 (Advanced): 3-Order Rolling Moving Average
**Problem Statement**: Compute a 3-order moving average of unit_price for each product over time (1 preceding, current, 1 following).  
**Input / Context**: orders DataFrame

In [ ]:
# Exercise 3: Rolling moving average

---

## Production Debugging Exercise

### Non-deterministic deduplication

**Scenario**: A dedup job uses `row_number().over(partitionBy(id).orderBy(updated_at))` and sometimes keeps different rows.

**Your task**: identify the most likely root cause, propose one diagnostic step, and propose
one fix. Write your answer before opening the model approach.

**Hints**

1. Check for duplicate `updated_at` values.
2. Add a deterministic tie-breaker such as a version or ingestion timestamp.
3. Validate the row count before and after.

<details>
<summary>Model approach (open after trying)</summary>

1. Reproduce the issue with the smallest possible dataset.
2. Use the Spark UI and the physical plan to locate the stage or operator that dominates.
3. Change one variable at a time and re-measure.
4. Record the before/after metric so the fix is defensible.

The exact fix depends on the evidence. The important habit is to separate allocation,
scheduling, shuffle, memory, and data-quality causes instead of guessing.

</details>

---

## Reference Solutions

<details>
<summary><b>Click to expand full solutions</b></summary>

#### Solution 1 (Top 1 Spending Customer per Region):
```python
cust_spend = orders.join(customers,"customer_id").groupBy("region","customer_id").agg(F.sum(F.col("quantity")*F.col("unit_price")).alias("spend"))
w = Window.partitionBy("region").orderBy(F.desc("spend"))
top_custs = cust_spend.withColumn("rn", F.row_number().over(w)).filter("rn = 1")
top_custs.show()
```

#### Solution 2 (Percentage of Total Spend per Customer):
```python
w_cust_all = Window.partitionBy("customer_id")
orders.withColumn("val", F.col("quantity")*F.col("unit_price")) \
      .withColumn("total_cust_spend", F.sum("val").over(w_cust_all)) \
      .withColumn("pct_of_total", F.round((F.col("val") / F.col("total_cust_spend")) * 100, 2)) \
      .select("customer_id","order_id","val","pct_of_total").show(5)
```

#### Solution 3 (3-Order Rolling Moving Average):
```python
w_roll = Window.partitionBy("product_id").orderBy("order_ts").rowsBetween(-1, 1)
orders.withColumn("moving_avg_price", F.round(F.avg("unit_price").over(w_roll), 2)) \
      .select("product_id","order_ts","unit_price","moving_avg_price").show(6)
```

</details>

---

## Module Summary & Interview Guide

### Key Takeaways
- **Window Power**: Performs aggregations without collapsing rows into a single record.
- **Ranking Functions**: row_number() is unique; rank() skips numbers on ties; dense_rank() does not skip.
- **Lead and Lag**: Enables time-series difference calculations without self-joins.

### Essential Technical Terms
- `Window.partitionBy`: Defines the boundary of rows processed together.
- `rowsBetween`: Defines physical row offset frame.
- `rangeBetween`: Defines logical value range frame.

### Common Engineering Mistakes
- **Window without partitionBy on Large Data**: Forces all data in the entire dataset into a single partition on 1 executor, causing OOM.
- **Confusing rank() with dense_rank()**: Rank leaves gaps after ties; dense_rank is sequential.

### Interview Questions on This Topic
- **Q: What happens if you omit partitionBy() in a Window?**
  - *A*: Spark moves the entire dataset into a single partition on one executor, completely disabling parallelism and likely crashing with OOM.
- **Q: What is the difference between lag() and lead()?**
  - *A*: lag() looks at preceding rows; lead() looks at subsequent rows.


### More Interview Questions

- **Q: What is the difference between `row_number`, `rank`, and `dense_rank`?**
  - *A*: `row_number` assigns unique sequential numbers; `rank` gives ties the same rank and leaves gaps; `dense_rank` gives ties the same rank without gaps.
- **Q: Why do window functions require a shuffle?**
  - *A*: Rows that belong to the same window partition must be co-located on the same task, so Spark shuffles by the partition keys and sorts within each partition.

### What to Study Next
Open 03_dataframe_and_spark_sql/07_nulls_and_data_quality.ipynb to master dirty record handling.

---

### Tear Down SparkSession
Always close your session when finishing to release executor threads and driver ports.

In [ ]:
spark.stop()
print('SparkSession successfully stopped.')